# Memoria a corto plazo en agentes de IA

## ¿De qué trata este notebook?

Este notebook profundiza en uno de los problemas más importantes de los agentes de IA: **la falta de memoria entre mensajes**.

Por defecto, cada mensaje que envías a un LLM es independiente. El modelo no recuerda lo que dijiste antes. Esto hace que los agentes sean imposibles de usar en conversaciones reales.

Aquí aprenderás a:
1. Ver el problema de memoria en acción
2. Resolverlo con `InMemorySaver`
3. Gestionar múltiples conversaciones independientes en paralelo
4. Inspeccionar el historial guardado
5. Recortar conversaciones largas para no exceder los límites del modelo

## La analogía

Sin memoria: cada llamada al agente es como llamar a un call center donde **nadie tiene tu historial**. Cada agente empieza desde cero, sin saber nada de ti.

Con memoria a corto plazo: es como hablar con un **mismo agente en una misma llamada** que recuerda todo lo que dijiste en esa sesión.

```
Sin memoria:                    Con memoria (short-term):
─────────────────               ─────────────────────────────
Usuario: "Me llamo Ana"         Usuario: "Me llamo Ana"
Agente:  "Hola, ¿en qué..."    Agente:  "Hola Ana, ¿en qué..."
                                        ↑ guarda en estado
Usuario: "¿Cómo me llamo?"     Usuario: "¿Cómo me llamo?"
Agente:  "No sé tu nombre"     Agente:  "Te llamas Ana"
                                        ↑ recupera del estado
```

---

## Paso 0 — Instalación y configuración

Necesitamos tres librerías:
- `langchain`: el framework principal de agentes
- `langchain-google-genai`: el conector con el modelo Gemini de Google
- `langgraph`: el motor que gestiona el estado y la memoria de los agentes

In [ ]:
# Instalar si no están disponibles
# !pip install -q langchain langchain-google-genai langgraph

In [1]:
from dotenv import load_dotenv
import os

# Carga las variables del archivo .env (donde guardamos la API Key de forma segura)
load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY_001")

# Si usas Google Colab, descomenta la línea de abajo en su lugar:
# API_KEY = userdata.get('GEMINI_API_KEY')

---

## Sección 1 — El problema: agente sin memoria

### ¿Qué va a pasar?

Vamos a crear un agente, hablar con él en dos turnos:
1. **Turno 1**: el usuario se presenta
2. **Turno 2**: el usuario pregunta cómo se llama

El resultado esperado sin memoria: el agente **no recordará el nombre** del turno 1 porque cada llamada a `invoke()` es completamente independiente.

### ¿Por qué cada llamada es independiente?

Técnicamente, un LLM no tiene estado. Cada vez que llamas a `invoke()`, solo recibe lo que le pasas en ese momento. Si no le pasas el historial de mensajes anteriores, no los conoce.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# Creamos el modelo con temperatura 0 (respuestas precisas y consistentes)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0)

In [ ]:
# Agente SIN checkpointer → sin memoria
# Cada llamada a invoke() es un mundo aparte
agente_sin_memoria = create_agent(
    model=llm,
    tools=[]  # Sin herramientas, solo conversación
)

print("=== AGENTE SIN MEMORIA ===")

print("\nTurno 1: Presentación")
# Primera llamada: el usuario se presenta
r1 = agente_sin_memoria.invoke({"messages": [HumanMessage("Hola, me llamo Carlos y soy del máster de IA.")]})
print("Agente:", r1["messages"][-1].content)

print("\nTurno 2: ¿Recuerda el nombre?")
# Segunda llamada: completamente nueva, no tiene contexto del turno 1
# RESULTADO ESPERADO: el agente dirá que no sabe el nombre
r2 = agente_sin_memoria.invoke({"messages": [HumanMessage("¿Cómo me llamo?")]})
print("Agente:", r2["messages"][-1].content)

---

## Sección 2 — La solución: InMemorySaver

### ¿Qué es un checkpointer?

Un checkpointer es un componente que **guarda y restaura el estado** de la conversación automáticamente. Funciona como el autoguardado de un videojuego: después de cada turno, guarda dónde estás. En el próximo turno, empieza desde donde lo dejó.

### ¿Qué es un thread?

Un **thread** es simplemente una conversación identificada con un nombre único (`thread_id`). Puedes tener:
- Thread "carlos" → conversación con Carlos
- Thread "maria" → conversación con María
- Thread "sesion-del-lunes" → conversación de la sesión del lunes

Cada thread tiene su propio historial independiente. Que Carlos diga su nombre en su thread no afecta a la conversación de María en su thread.

### ¿Cómo funciona técnicamente?

Antes de cada `invoke()`, el checkpointer:
1. Busca si hay mensajes guardados para ese `thread_id`
2. Si los hay, los añade al contexto
3. El modelo recibe todos los mensajes (antiguos + nuevo)
4. Después del `invoke()`, guarda el historial actualizado

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# Creamos el checkpointer — guarda el estado de TODAS las conversaciones en memoria RAM
# Un único checkpointer puede gestionar múltiples threads (conversaciones)
checkpointer = InMemorySaver()

# Agente CON memoria a corto plazo
# La única diferencia con el agente anterior: checkpointer=checkpointer
agente_con_memoria = create_agent(
    model=llm,
    tools=[],
    checkpointer=checkpointer  # ← Esta línea activa la memoria
)

# El thread_id identifica esta conversación específica
# Puedes usar cualquier string: un UUID, un nombre de usuario, un número de sesión...
config_thread1 = {"configurable": {"thread_id": "conversacion-1"}}

print("=== AGENTE CON MEMORIA (Thread 1) ===")

print("\nTurno 1: Presentación")
r1 = agente_con_memoria.invoke(
    {"messages": [HumanMessage("Hola, me llamo Carlos y soy del máster de IA.")]},
    config=config_thread1  # ← Asociamos este mensaje al thread "conversacion-1"
)
print("Agente:", r1["messages"][-1].content)

print("\nTurno 2: ¿Recuerda el nombre? (mismo thread)")
r2 = agente_con_memoria.invoke(
    {"messages": [HumanMessage("¿Cómo me llamo y qué estudio?")]},
    config=config_thread1  # ← Mismo thread_id = el agente tiene acceso al turno 1
)
print("Agente:", r2["messages"][-1].content)

---

## Sección 3 — Múltiples threads: conversaciones independientes

### ¿Por qué es importante?

En un sistema real con múltiples usuarios, necesitas que cada conversación sea privada e independiente. Lo que dice Carlos en su conversación no debe "contaminar" la conversación de María.

### ¿Cómo funciona?

Simplemente usando un `thread_id` diferente para cada conversación. El mismo `checkpointer` y el mismo `agente_con_memoria` gestionan todos los threads, pero cada uno tiene su propio historial separado.

In [ ]:
# Thread 2: una conversación completamente nueva, con un usuario diferente
# Nota: usamos el MISMO agente (agente_con_memoria) y el MISMO checkpointer
# Lo que cambia es el thread_id
config_thread2 = {"configurable": {"thread_id": "conversacion-2"}}

print("=== THREAD 2 (conversación diferente) ===")

# María empieza su propia conversación en el thread 2
r_thread2 = agente_con_memoria.invoke(
    {"messages": [HumanMessage("Hola, me llamo María.")]},
    config=config_thread2  # ← Thread diferente = conversación diferente
)
print("María - Turno 1:", r_thread2["messages"][-1].content)

# Volvemos al thread de Carlos: ¿sigue recordando?
# Esto demuestra que los threads son completamente independientes
r_carlos_again = agente_con_memoria.invoke(
    {"messages": [HumanMessage("¿Me recuerdas?")]},
    config=config_thread1  # ← Thread de Carlos — el agente sigue teniendo su historial
)
print("\nCarlos en Thread 1 - ¿Me recuerdas?:")
print("Agente:", r_carlos_again["messages"][-1].content)

---

## Sección 4 — Inspeccionar el estado del thread

### ¿Para qué sirve?

El método `get_state()` te permite ver todos los mensajes guardados en un thread en cualquier momento. Es útil para:
- Depurar problemas (ver exactamente qué historial tiene el agente)
- Auditar conversaciones
- Saber cuántos mensajes lleva la conversación (para decidir si hay que recortarlos)

### ¿Qué devuelve?

Un objeto de estado con una lista de todos los mensajes guardados para ese thread. Cada mensaje tiene un tipo (HumanMessage = usuario, AIMessage = agente) y su contenido.

In [ ]:
# Recuperamos el estado completo del Thread 1 (la conversación de Carlos)
# get_state() no envía ningún mensaje nuevo, solo lee el historial guardado
estado = agente_con_memoria.get_state(config_thread1)

print("📋 Historial del Thread 1 (Carlos):")
print(f"   Total de mensajes: {len(estado.values['messages'])}\n")

# Iteramos sobre todos los mensajes guardados y los mostramos
for i, msg in enumerate(estado.values["messages"], 1):
    # Distinguimos entre mensajes del usuario y del agente por el tipo de objeto
    rol = "👤 Usuario" if msg.__class__.__name__ == "HumanMessage" else "🤖 Agente"
    # Truncamos en 100 caracteres para no saturar la pantalla
    print(f"  [{i}] {rol}: {msg.content[:100]}")

---

## Sección 5 — Gestión de conversaciones largas: Trim Messages

### El problema que resuelve

Los LLMs tienen un límite en la cantidad de texto que pueden procesar de una vez. Este límite se llama **ventana de contexto** y se mide en tokens (fragmentos de texto).

Si una conversación tiene 200 mensajes y cada uno tiene 100 palabras, el historial completo puede exceder esa ventana. En ese caso, el modelo daría un error o ignoraría los mensajes más antiguos de forma no controlada.

### La solución: recortar mensajes

Cuando la conversación se vuelve muy larga, podemos eliminar los mensajes más antiguos automáticamente. Esto mantiene el agente funcional dentro de los límites del modelo.

### Estrategias de gestión de memoria

| Estrategia | Cuándo usarla | Ventaja | Desventaja |
|------------|---------------|---------|------------|
| **Sin recorte** | Conversaciones cortas | Contexto completo | Puede exceder ventana |
| **Trim** | Conversaciones largas simples | Rápido y simple | Pierde contexto antiguo |
| **Summarize** | Conversaciones largas importantes | Retiene info clave | Más lento (llamada extra al LLM) |
| **Delete** | Limpiar mensajes específicos | Control granular | Requiere lógica manual |

In [ ]:
from langchain.agents import AgentState
from langchain_core.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime
from typing import Any

# Esta función crea un middleware de recorte de mensajes
# n → cuántos mensajes queremos conservar (los más recientes)
def trim_to_last_n_messages(n: int = 4):
    """Crea un middleware que mantiene solo los últimos N mensajes."""

    # Esta función interna es la que realmente se ejecuta antes de cada llamada al modelo
    def trim(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        messages = state["messages"]

        # Si el número de mensajes es menor o igual a N, no hacemos nada
        if len(messages) <= n:
            return None  # None indica que no hay cambios que hacer

        print(f"  ✂️ Recortando: {len(messages)} → {n} mensajes")

        # Guardamos solo los últimos N mensajes
        recent = messages[-n:]

        # Devolvemos un estado que:
        # 1. Elimina TODOS los mensajes actuales (RemoveMessage con REMOVE_ALL_MESSAGES)
        # 2. Añade solo los N mensajes más recientes
        return {
            "messages": [
                RemoveMessage(id=REMOVE_ALL_MESSAGES),
                *recent  # El asterisco desempaqueta la lista en elementos individuales
            ]
        }

    return trim

print("✅ Función de trim definida")
print("💡 En producción, se usa como middleware en create_agent()")
print("   Ver: https://docs.langchain.com/oss/python/langchain/short-term-memory#trim-messages")

---

## Ejercicio propuesto

1. Crea una conversación de al menos 6 turnos sobre un tema de tu elección
2. Después de cada turno, usa `get_state()` para ver cómo crece el historial
3. Implementa un trim que conserve el primer mensaje (presentación) y los últimos 2
4. Bonus: crea 3 threads con 3 "usuarios" distintos y comprueba que sus historiales son independientes

## Resumen

| Concepto | Descripción | Cómo usarlo |
|----------|-------------|-------------|
| **Sin memoria** | Cada `invoke()` es independiente | `create_agent(model, tools=[])` |
| **InMemorySaver** | Guarda el historial en RAM | `checkpointer=InMemorySaver()` |
| **thread_id** | Identifica una conversación | `config={"configurable": {"thread_id": "..."}}` |
| **get_state()** | Ver el historial guardado | `agente.get_state(config)` |
| **Trim** | Recortar mensajes antiguos | Middleware con `RemoveMessage` |